# Factor Pricing: High-Dimensional Factor Extraction and Dynamic Asset Pricing

**Full pipeline notebook** — covers all modules including the roadmap extensions:

1. Setup & data loading
2. Exploratory data analysis
3. Latent factor extraction (PCA + Bai-Ng criterion)
4. Observed factor model (Fama-French 5 + macro)
5. Forecasting experiments (OLS / Ridge / Lasso / ElasticNet / XGBoost / LightGBM)
6. LSTM nonlinear forecaster
7. Dynamic models: Kalman smoother + Multivariate DFM
8. Model evaluation with multiple-testing corrections
9. Cross-sectional quintile prediction
10. Portfolio construction and backtesting

**Google Colab quick-start** — run the first cell to clone the repo and install deps.


In [ ]:
# ── Run this cell only on Google Colab ────────────────────────────────────────
import sys, os

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !git clone -b claude/factor-pricing-project-lTbTV \
        https://github.com/meteor21/nn-model-for-tennis.git repo
    %cd repo/factor-pricing
    !pip install -q yfinance pandas_datareader scikit-learn statsmodels xgboost lightgbm torch
else:
    # Local: make sure we're in the factor-pricing root
    project_root = os.path.abspath('..')
    if project_root not in sys.path:
        sys.path.insert(0, project_root)
    os.chdir(project_root)

print("Working directory:", os.getcwd())


## 1. Setup

In [ ]:
import sys, os, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.rcParams['figure.figsize'] = (12, 5)
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

import config
RESULTS_DIR = 'results'
DATA_DIR    = 'data'
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(DATA_DIR,    exist_ok=True)

print('Config loaded.')
print('  Date range  :', config.START_DATE, '–', config.END_DATE)
print('  N_FACTORS   :', config.N_FACTORS)
print('  ROLLING_WINDOW:', config.ROLLING_WINDOW)


## 2. Data Loading

In [ ]:
from src.data_pipeline import (
    download_equity_data, download_etf_data,
    download_ff_factors, download_fred_data,
    build_panel, save_panel, load_panel,
)

# Optional: set your FRED API key here (or export FRED_API_KEY env var)
# import os; os.environ['FRED_API_KEY'] = 'YOUR_KEY_HERE'

PANEL_PATH   = os.path.join(DATA_DIR, config.PANEL_FILE)
FORCE_DOWNLOAD = False   # set True to re-download

if not FORCE_DOWNLOAD and os.path.exists(PANEL_PATH):
    panel = load_panel(PANEL_PATH)
    print('Loaded cached panel:', panel.shape)
else:
    equity_prices = download_equity_data(config.EQUITY_TICKERS, config.START_DATE, config.END_DATE)
    etf_prices    = download_etf_data(config.SECTOR_ETFS,       config.START_DATE, config.END_DATE)
    ff_factors    = download_ff_factors(config.START_DATE, config.END_DATE)
    fred_key      = os.environ.get('FRED_API_KEY')
    macro_df      = download_fred_data(config.START_DATE, config.END_DATE, api_key=fred_key)
    panel = build_panel(equity_prices, etf_prices, ff_factors, macro_df)
    save_panel(panel, PANEL_PATH)
    print('Panel built and saved:', panel.shape)

panel.head(3)


In [ ]:
# ── Synthetic data fallback (use when no internet / no API key) ───────────────
# Uncomment and run if the cell above fails.

# from generate_synthetic import generate_synthetic_panel
# panel = generate_synthetic_panel(n_assets=30, n_days=756)
# print('Synthetic panel shape:', panel.shape)


## 3. Exploratory Data Analysis

In [ ]:
from src.eda import (
    plot_correlation_heatmap, plot_eigenvalue_scree,
    plot_cumulative_variance, summary_stats,
)

ret_cols = panel.columns[panel.columns.str.endswith('_ret')]
returns  = panel[ret_cols]

stats = summary_stats(returns)
print('Summary statistics (first 8 assets):')
stats.head(8)


In [ ]:
plot_correlation_heatmap(returns, results_dir=RESULTS_DIR)
plot_eigenvalue_scree(returns, n_components=20, results_dir=RESULTS_DIR)
plot_cumulative_variance(returns, n_components=20, results_dir=RESULTS_DIR)
print('EDA plots saved to', RESULTS_DIR)


## 4. Latent Factor Extraction (PCA + Bai-Ng)

In [ ]:
from src.factor_models import LatentFactorModel, bai_ng_criterion, ObservedFactorModel

bn = bai_ng_criterion(returns, max_factors=config.MAX_FACTORS, results_dir=RESULTS_DIR)
print('Bai-Ng optimal k:', bn)


In [ ]:
lfm = LatentFactorModel(n_factors=config.N_FACTORS)
lfm.fit(returns)
latent_factors = lfm.transform(returns)
print('Latent factor returns shape:', latent_factors.shape)
print('Explained variance per factor (%):', np.round(lfm.explained_variance_ratio_ * 100, 2))

latent_factors.plot(subplots=True, figsize=(14, 10), title='Latent Factor Returns')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'latent_factor_returns.png'), dpi=150)
plt.show()


## 5. Observed Factor Model (FF5 + Macro)

In [ ]:
ff5_cols   = [c for c in ['Mkt-RF','SMB','HML','RMW','CMA','RF'] if c in panel.columns]
macro_cols = [c for c in config.FRED_SERIES.keys() if c in panel.columns]

observed_factors = panel[ff5_cols + macro_cols].copy()

ofm = ObservedFactorModel(add_constant=True)
ofm.fit(returns, observed_factors)

betas = ofm.get_factor_exposures()
print('Beta matrix shape:', betas.shape)

fig, ax = plt.subplots(figsize=(16, 6))
sns.heatmap(betas.T, cmap='RdBu_r', center=0, linewidths=0.3, ax=ax)
ax.set_title('Observed Factor Betas (factors × assets)')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'observed_factor_betas.png'), dpi=150)
plt.show()


## 6. Forecasting Experiments

Trains OLS, Ridge, Lasso, ElasticNet, and optionally XGBoost / LightGBM on
lagged factor + macro features.  Uses `TimeSeriesSplit` cross-validation for
regularisation parameter selection.


In [ ]:
from src.regression_models import run_forecasting_experiment

macro_df = panel[macro_cols] if macro_cols else pd.DataFrame(index=panel.index)

forecast_results = run_forecasting_experiment(
    panel_df         = panel,
    latent_factors   = latent_factors,
    observed_factors = observed_factors,
    macro_df         = macro_df,
    train_ratio      = config.TRAIN_RATIO,
    val_ratio        = config.VAL_RATIO,
    lags             = config.FEATURE_LAGS,
)
print('Models trained:', list(forecast_results.keys()))


In [ ]:
from src.evaluation import compare_models_table

for split in ('train', 'val', 'test'):
    compare_models_table(forecast_results, split=split)


## 7. LSTM Nonlinear Forecaster

Walk-forward LSTM (PyTorch) with early stopping. Falls back to
sklearn `MLPRegressor` when PyTorch is unavailable.


In [ ]:
from src.nonlinear_models import rolling_lstm_forecast

# Equal-weighted portfolio return as the target
ew_target = returns.mean(axis=1).rename('ew_return')

lstm_preds, lstm_actuals = rolling_lstm_forecast(
    factor_returns = latent_factors,
    macro_df       = macro_df,
    target         = ew_target,
    window         = config.ROLLING_WINDOW,
    seq_len        = 21,
    step           = 21,          # refit every 21 trading days
    hidden_size    = 32,
    epochs         = 30,
)
print(f'LSTM predictions: {len(lstm_preds)} dates')


In [ ]:
from src.evaluation import compute_metrics, plot_predictions_vs_actuals

lstm_m = compute_metrics(lstm_actuals.values, lstm_preds.values)
print('LSTM walk-forward metrics:')
for k, v in lstm_m.items():
    print(f'  {k:20s}: {v:.4f}')

plot_predictions_vs_actuals(
    lstm_actuals, lstm_preds,
    title='LSTM: Walk-Forward Predictions vs Actuals',
    results_dir=RESULTS_DIR,
)


## 8. Dynamic Models

### 8a. Kalman Factor Smoother (univariate local-level)
### 8b. Multivariate Dynamic Factor Model (EM)


In [ ]:
from src.dynamic_models import KalmanFactorModel

kf = KalmanFactorModel(n_iter=20)
kf.fit(latent_factors)
smoothed = kf.smooth()

fig, axes = plt.subplots(config.N_FACTORS, 1, figsize=(14, 3 * config.N_FACTORS), sharex=True)
for i, col in enumerate(latent_factors.columns):
    axes[i].plot(latent_factors.index, latent_factors[col],
                 alpha=0.4, label='Raw', color='steelblue')
    axes[i].plot(smoothed.index, smoothed[col],
                 label='Kalman-smoothed', color='darkorange', linewidth=1.5)
    axes[i].set_title(col, fontsize=9)
    axes[i].legend(fontsize=7)
plt.suptitle('Kalman Smoothing of Latent Factors', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'kalman_smoothed_factors.png'), dpi=150)
plt.show()


In [ ]:
from src.dynamic_models import MultivariateDFM

dfm = MultivariateDFM(n_iter=30)
dfm.fit(latent_factors)
dfm_smoothed = dfm.smooth()

# Impulse-response for a shock to the first factor
irf = dfm.impulse_response(shock_factor=0, horizon=20)
fig, axes = plt.subplots(1, config.N_FACTORS, figsize=(14, 4), sharey=False)
for k in range(config.N_FACTORS):
    axes[k].plot(irf[:, k], marker='o', markersize=3)
    axes[k].axhline(0, color='grey', linewidth=0.7, linestyle='--')
    axes[k].set_title(f'Response of F{k+1}')
    axes[k].set_xlabel('Horizon (days)')
fig.suptitle('Impulse Response to Shock on Factor 1', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'impulse_response.png'), dpi=150)
plt.show()
print('Transition matrix A:')
print(pd.DataFrame(dfm.A_, columns=latent_factors.columns, index=latent_factors.columns).round(3))


## 9. Model Evaluation & Multiple-Testing Corrections

After running multiple models on the same test set the probability of
spurious significance is inflated.  We apply:

- **Bonferroni** — controls family-wise error rate (FWER)
- **Benjamini-Hochberg** — controls false-discovery rate (FDR)
- **Model Confidence Set** (Hansen et al. 2011) — iterative bootstrap


In [ ]:
from src.evaluation import (
    diebold_mariano_test, plot_rolling_r2,
    multiple_testing_correction, model_confidence_set,
)

# Collect test-set predictions from the static forecasting experiments
actuals_test = None
pred_dict    = {}
for name, res in forecast_results.items():
    if 'test' in res and res['test'].get('predictions') is not None:
        pred_s = pd.Series(res['test']['predictions'])
        act_s  = pd.Series(res['test']['actuals'])
        pred_dict[name] = pred_s
        if actuals_test is None:
            actuals_test = act_s

print('Models in test set:', list(pred_dict.keys()))


In [ ]:
# Diebold-Mariano test for each model vs naive (zero) forecast
dm_results = {}
if actuals_test is not None:
    for name, pred in pred_dict.items():
        e_model = (actuals_test - pred).dropna().values
        e_naive = actuals_test.iloc[:len(e_model)].values
        dm = diebold_mariano_test(e_model, e_naive)
        dm_results[name] = dm
    dm_df = pd.DataFrame(dm_results).T
    print(dm_df[['dm_stat', 'p_value']].round(4))


In [ ]:
# Multiple-testing correction on DM p-values
if dm_results:
    p_vals = {k: v['p_value'] for k, v in dm_results.items()}
    mc = multiple_testing_correction(p_vals, alpha=0.05)
    print('\nMultiple-testing correction results:')
    print(pd.DataFrame(mc).T.round(4))


In [ ]:
# Model Confidence Set (bootstrap)
if actuals_test is not None and len(pred_dict) >= 2:
    # Build loss matrix: T × M  (squared errors)
    names = list(pred_dict.keys())
    min_len = min(len(p) for p in pred_dict.values())
    act_arr = actuals_test.iloc[:min_len].values
    loss_matrix = np.column_stack([
        (act_arr - pred_dict[n].values[:min_len]) ** 2
        for n in names
    ])
    loss_df = pd.DataFrame(loss_matrix, columns=names)
    mcs_survivors = model_confidence_set(loss_df, alpha=0.10, n_bootstrap=500)
    print('Model Confidence Set (alpha=10%):', mcs_survivors)


In [ ]:
# Rolling R² plot comparing all models
if actuals_test is not None and pred_dict:
    plot_rolling_r2(pred_dict, actuals_test, window=63, results_dir=RESULTS_DIR)
    print('Rolling R² plot saved.')


## 10. Cross-Sectional Quintile Prediction

Shifts from aggregate portfolio forecasting to **asset-level rank prediction**:

- Estimates each asset's factor betas via rolling OLS
- Predicts next-period return using beta × lag-1 factor forecast
- Ranks assets into quintiles (1 = worst, 5 = best)
- Evaluates via **Rank IC** (Spearman ρ) and long-short quintile spread


In [ ]:
from src.cross_sectional import CrossSectionalPredictor, summarise_cross_sectional

cs_pred = CrossSectionalPredictor(window=252, n_quintiles=5)
cs_results = cs_pred.fit_predict(
    returns_df     = returns,
    factor_returns = latent_factors,
    step           = 21,
)
print('Cross-sectional results shape:', cs_results.shape)
print(cs_results.head())


In [ ]:
summary_cs = summarise_cross_sectional(cs_pred)
cs_pred.plot_quintile_returns(results_dir=RESULTS_DIR)
print('Quintile return plot saved.')


In [ ]:
# Long-short spread cumulative return
ls = cs_pred.long_short_spread()
(1 + ls).cumprod().plot(figsize=(12, 4),
    title='Cumulative Return: Long Q5 / Short Q1')
plt.axhline(1, color='grey', linewidth=0.8, linestyle='--')
plt.ylabel('Cumulative return (×)')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'ls_cumulative.png'), dpi=150)
plt.show()


## 11. Portfolio Construction and Backtesting

In [ ]:
from src.portfolio import (
    mean_variance_portfolio, factor_mimicking_portfolio,
    backtest_portfolio, compare_portfolios,
)

latent_w_mat = factor_mimicking_portfolio(returns, latent_factors)
obs_w_mat    = factor_mimicking_portfolio(returns, observed_factors[ff5_cols])

latent_weights = latent_w_mat.iloc[:, 0]
obs_weights    = obs_w_mat.iloc[:, 0] if not obs_w_mat.empty else pd.Series()

ew_weights = pd.Series(
    np.ones(len(returns.columns)) / len(returns.columns),
    index=returns.columns,
)
if obs_weights.empty or obs_weights.isna().all():
    obs_weights = ew_weights.copy()

summary_port = compare_portfolios(
    latent_factor_weights   = latent_weights,
    observed_factor_weights = obs_weights,
    equal_weights           = ew_weights,
    returns_df              = returns,
    results_dir             = RESULTS_DIR,
)
summary_port


In [ ]:
# Mean-variance portfolio
sample_ret = returns.iloc[-config.ROLLING_WINDOW:]
exp_ret    = sample_ret.mean()
cov_mat    = sample_ret.cov()
mv_w       = mean_variance_portfolio(exp_ret, cov_mat, risk_aversion=config.RISK_AVERSION)
mv_result  = backtest_portfolio(mv_w, returns)
print('Mean-Variance Portfolio:')
for k, v in mv_result.items():
    if not isinstance(v, pd.Series):
        print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')


## 12. Results Summary

In [ ]:
print('=' * 60)
print('RESULTS SUMMARY')
print('=' * 60)

# Static forecasting
print('\n── Static Forecasting (test set) ──────────────────────')
for name, res in forecast_results.items():
    if 'test' in res:
        m = res['test'].get('metrics', {})
        r2 = m.get('r2', float('nan'))
        sh = m.get('sharpe', float('nan'))
        print(f'  {name:20s}  R²={r2:.4f}  Sharpe={sh:.3f}')

# LSTM
print('\n── LSTM Walk-Forward ───────────────────────────────────')
for k, v in lstm_m.items():
    print(f'  {k:20s}: {v:.4f}')

# Cross-sectional
print('\n── Cross-Sectional Prediction ──────────────────────────')
print(f"  Mean Rank IC  : {summary_cs['mean_ic']:.4f}  (t={summary_cs['ic_t_stat']:.2f})")
print(f"  IC > 0        : {summary_cs['ic_positive_pct']:.1f}%")
print(f"  L/S Ann Return: {summary_cs['ls_ann_return']*100:.2f}%")
print(f"  L/S Sharpe    : {summary_cs['ls_sharpe']:.3f}")

print('\nAll result files saved to:', RESULTS_DIR)
print('Files:', sorted(os.listdir(RESULTS_DIR)))
